In [1]:
# from datasets import load_dataset|
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,
)
from peft import LoraConfig
from trl import SFTTrainer
import torch
import json
from peft import prepare_model_for_kbit_training

e:\AI-VENV\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [3]:
MODEL_NAME = "Qwen/Qwen3-8B"


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

In [5]:
bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_use_double_quant=True
)

In [7]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [3]:
from datasets import load_dataset
dataset = load_dataset(
    "json",
    data_files= "./dataset/V3A.jsonl"

)

ImportError: The pyarrow installation is not built with support for 'dataset' (DLL load failed while importing _fs: An Application Control policy has blocked this file.)

V3A FULL DATASET ANALYSIS

Dataset:
G:\V1-V16-dataset\dataset\v3a_dataset_clean.jsonl

File size: 99.88 MB

Scanning entire dataset...
--------------------------------------------------------------------------------

ANALYSIS COMPLETE

Total records       : 67,504
Valid JSON          : 67,504
Invalid JSON        : 0
Blank lines         : 0
Total issues        : 0

Understanding schemas: 1
Input schemas        : 1
Output schemas       : 1
Profile schemas      : 1

Report saved to:
G:\V1-V16-dataset\V3A_FULL_ANALYSIS_REPORT.txt

Problem records saved to:
G:\V1-V16-dataset\V3A_PROBLEM_RECORDS.jsonl



In [ ]:
# 

In [ ]:
import json

# ==========================================================
# Prepare Model
# ==========================================================

model = prepare_model_for_kbit_training(model)

model.enable_input_require_grads()
model.gradient_checkpointing_enable()
model.config.use_cache = False

# ==========================================================
# System Prompt
# ==========================================================

SYSTEM_PROMPT = """You are an expert assistant objective prediction model.

Your task is to predict the assistant's objective from the given conversation.

Rules:
- Predict only the assistant's objective.
- Do NOT generate a reply.
- Do NOT summarize the conversation.
- Do NOT extract memories.
- Do NOT infer unsupported information.
- Base your prediction only on the provided conversation.

Return ONLY valid JSON in the following format:

{
  "primary_objective": "",
  "secondary_objective": "",
  "priority": "",
  "reason": ""
}

Field Definitions:
- primary_objective: The assistant's main objective.
- secondary_objective: An optional supporting objective. Use "None" if no meaningful secondary objective exists.
- priority: One of "High", "Medium", or "Low".
- reason: A brief explanation (1–2 sentences) describing why these objectives were selected.

Return only the JSON object. Do not include any extra text or markdown.
"""

# ==========================================================
# Formatting Function
# ==========================================================

def formatting_func(example):

    user_input = (
        f"{example['instruction']}\n\n"
        f"{json.dumps(example['input'], ensure_ascii=False, separators=(',', ':'))}"
    )

    assistant_output = json.dumps(
        example["output"],
        ensure_ascii=False,
        separators=(",", ":")
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_input,
        },
        {
            "role": "assistant",
            "content": assistant_output,
        },
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

In [ ]:
# Printing  TOKEN ,DISTRIBUTION ,FORMATTING ,LONGEST ,SAMPLES of this dataset (important thing before training)

import time
from statistics import mean, median

def analyze_dataset(name, dataset, tokenizer, formatting_func):
    print("\n" + "="*70)
    print(name)
    print("="*70)

    train = dataset["train"]

    lengths = []
    format_times = []

    start_total = time.time()

    for i, sample in enumerate(train):
        t1 = time.time()

        text = formatting_func(sample)

        t2 = time.time()
        format_times.append(t2 - t1)

        tokens = tokenizer(text, add_special_tokens=True)["input_ids"]
        lengths.append(len(tokens))

        if (i + 1) % 5000 == 0:
            print(f"Processed {i+1}/{len(train)}")

    total_time = time.time() - start_total

    print("\n----- BASIC -----")
    print("Samples              :", len(train))
    print("Average Tokens       :", round(mean(lengths),2))
    print("Median Tokens        :", median(lengths))
    print("Minimum Tokens       :", min(lengths))
    print("Maximum Tokens       :", max(lengths))

    print("\n----- TOKEN DISTRIBUTION -----")
    print(">256 tokens          :", sum(x > 256 for x in lengths))
    print(">512 tokens          :", sum(x > 512 for x in lengths))
    print(">1024 tokens         :", sum(x > 1024 for x in lengths))
    print(">2048 tokens         :", sum(x > 2048 for x in lengths))
    print(">4096 tokens         :", sum(x > 4096 for x in lengths))

    print("\n----- FORMATTING -----")
    print("Formatting Time      :", round(total_time,2), "sec")
    print("Average/sample       :", round(mean(format_times)*1000,3), "ms")
    print("Samples/sec          :", round(len(train)/total_time,2))

    print("\n----- LONGEST SAMPLES -----")
    top = sorted(enumerate(lengths), key=lambda x: x[1], reverse=True)[:10]

    for idx, tok in top:
        print(f"Sample {idx:6d} : {tok} tokens")

    return lengths


In [ ]:
old_dataset = load_dataset(
    "json",
    data_files= r"./v5a/V5A.jsonl",
  
)

# new_dataset = load_dataset(
#     "json",
#     data_files=r"./newV4a/finalV4A.jsonl",
  
# )


old_lengths = analyze_dataset(
    "OLD DATASET",
    old_dataset,
    tokenizer,
    formatting_func
)

# new_lengths = analyze_dataset(
#     "NEW V4A DATASET",
#     new_dataset,
#     tokenizer,
#     formatting_func
# )


OLD DATASET
Processed 5000/59921
Processed 10000/59921
Processed 15000/59921
Processed 20000/59921
Processed 25000/59921
Processed 30000/59921
Processed 35000/59921
Processed 40000/59921
Processed 45000/59921
Processed 50000/59921
Processed 55000/59921

----- BASIC -----
Samples              : 59921
Average Tokens       : 446.63
Median Tokens        : 439
Minimum Tokens       : 303
Maximum Tokens       : 879

----- TOKEN DISTRIBUTION -----
>256 tokens          : 59921
>512 tokens          : 10744
>1024 tokens         : 0
>2048 tokens         : 0
>4096 tokens         : 0

----- FORMATTING -----
Formatting Time      : 87.0 sec
Average/sample       : 0.23 ms
Samples/sec          : 688.73

----- LONGEST SAMPLES -----
Sample  49784 : 879 tokens
Sample  24899 : 877 tokens
Sample  54440 : 850 tokens
Sample  26312 : 848 tokens
Sample  17506 : 842 tokens
Sample   9744 : 836 tokens
Sample  17119 : 833 tokens
Sample  39085 : 819 tokens
Sample  48582 : 818 tokens
Sample   4658 : 808 tokens
